<!-- ## Topics we will be covering -->

**Part A - Basics**
1. How a computer runs your code
    - CPU, core, RAM, operating system
    - Program, process and thread
2. Key ideas
    - Sequential vs concurrent vs parallel
    - I/O-bound vs CPU-bound work
    - Measuring time with `time.perf_counter()`
3. Multithreading basics
    - Your first thread: `Thread(target, args)`, `start()`, `join()`
    - Many threads with a loop, common mistakes
    - The GIL - why threads don't speed up calculations
    - Race condition and `Lock`
4. Multiprocessing basics
    - Your first process and the `if __name__ == "__main__":` guard
    - Multiprocessing inside Jupyter on Windows
    - Speeding up calculations, processes don't share memory
5. Pool Executors (`concurrent.futures`)
    - `ThreadPoolExecutor` and `ProcessPoolExecutor`
    - `submit()`, `result()`, `map()`, `as_completed()`, handling errors
    - Which one should I use?

**Part B - Advanced (optional)**

6. More about threads - a thread as a class, getting results back, `kwargs`, daemon threads, `Timer`
7. More synchronization tools - deadlock, `RLock`, `Semaphore`, `Event`, `Condition`, `queue.Queue` (producer-consumer)
8. More about multiprocessing - start methods, `multiprocessing.Queue`, `Value` / `Array`, `Pool`, `Pipe`, `Manager`, `terminate()`
9. The future of the GIL and `asyncio`

**Practice questions and a mini project**

> **How to use this notebook**
> * Run the cells from top to bottom - later cells use names created by earlier ones.
> * Keep the file **`session_4_workers.py`** in the same folder as this notebook. The multiprocessing examples need it (section 4 explains why).
> * Click a **▶ summary** line to open the explanation or answer hidden under it.
> * **Quick check** boxes are small quizzes - think of the answer first, then open it.
> * **Part A** is the main session. **Part B** is optional - read it once Part A is clear.
> * Timings depend on your computer. Your numbers will be different, but the pattern will be the same.
> * The output of threads can come in a **different order every time** you run a cell. That is normal.

---
# Part A - Basics

### 1. How a Computer Runs Your Code

So far, every program we wrote did **one thing at a time**: line 1, then line 2, then line 3. In this session we learn how to do **several things at the same time**. First we need four simple words.

<details>
<summary> 1. <b>CPU (processor)</b></summary>
The CPU is the "brain" of the computer. It runs the instructions of our programs - adding numbers, comparing values and so on. It is very fast: billions of small steps every second.
</details>

<details>
<summary> 2. <b>Core</b></summary>
A CPU has several <b>cores</b>. Each core is like a small CPU of its own and can do one job at a time. A CPU with 8 cores can really do 8 jobs at the same moment.

🏏 <b>Cricket example:</b> cores are like <b>bowlers in the nets</b>. One bowler can bowl to only one batsman at a time. With 4 bowlers, 4 batsmen can practise at the same time.
</details>

<details>
<summary> 3. <b>RAM (memory)</b></summary>
RAM is where a running program keeps its data. All our variables, lists and objects live in RAM while the program runs.
</details>

<details>
<summary> 4. <b>Operating system (OS)</b></summary>
The operating system (Windows, Linux, macOS) controls the computer. It decides which job runs on which core and for how long. It switches between jobs so quickly that it <i>looks</i> like everything runs at once.

🏏 <b>Cricket example:</b> the OS is like the <b>captain</b> who decides who bowls the next over.
</details>

##### How many cores does my computer have?
* In Python: `os.cpu_count()`
* In Windows: open the **Task Manager** (`Ctrl + Shift + Esc`) → **Performance** → **CPU**. Look at *Cores* and *Logical processors*.

In [ ]:
import os

print("CPU cores on this computer:", os.cpu_count())

##### Program, Process and Thread

<details>
<summary> 1. <b>Program</b></summary>
A program is a file with instructions, for example <code>scorer.py</code>. On its own it does nothing - it is just text saved on the disk.

🏏 <b>Cricket example:</b> the <b>team plan written on paper</b>.
</details>

<details>
<summary> 2. <b>Process</b></summary>
A process is a <b>running program</b>. When you run <code>python scorer.py</code>, the operating system creates a process. Every process has:
<ul>
<li>its <b>own memory</b> - no other process can see its variables,</li>
<li>its own copy of Python,</li>
<li>its own number, called the <b>process id (PID)</b>.</li>
</ul>
You can see all running processes in the Task Manager → <b>Details</b> tab.

🏏 <b>Cricket example:</b> a process is a <b>match being played in a stadium</b>. Two matches in two stadiums are completely separate - each has its own ground, ball and scoreboard.
</details>

<details>
<summary> 3. <b>Thread</b></summary>
A thread is a <b>worker inside a process</b> that runs code. Every process starts with one thread, called the <b>main thread</b>. All the code we wrote so far ran in the main thread. A process can create more threads, and <b>all threads of a process share the same memory</b> - they see the same variables.

🏏 <b>Cricket example:</b> threads are the <b>players on the field</b> in one stadium. They all share the same ground, ball and scoreboard. That is useful (everyone sees the same score), but also risky (two players changing the scoreboard at the same moment can make a mistake - we will see this soon).
</details>

```
 Computer
 ├── Process: python scorer.py   (PID 4120, own memory)
 │     ├── Main thread
 │     ├── Thread 2
 │     └── Thread 3          <- these threads share the memory of their process
 └── Process: chrome.exe         (PID 7788, own memory)
       ├── Main thread
       └── ...
```

In [ ]:
import threading

print("My process id (PID):", os.getpid())                          # the process running this notebook
print("My thread          :", threading.current_thread().name)     # the main thread

<details>
<summary><b>Quick check:</b> Two threads in the same process change the variable <code>score</code>. Do both see the new value? What about two processes?</summary>
Two <b>threads</b> - yes, they share the memory of their process. Two <b>processes</b> - no, each process has its own memory and its own copy of <code>score</code>.
</details>

### 2. Key Ideas

##### Sequential, concurrent and parallel
🏏 A coach in the nets has three batsmen who want throw-downs.

| Way of working | Cricket example | In Python |
| --- | --- | --- |
| **Sequential** | one coach: batsman 1 gets all his balls, **then** batsman 2, **then** batsman 3 | normal code |
| **Concurrent** | one coach **switches** between batsmen: while batsman 1 walks back to fetch the ball, the coach throws to batsman 2 | threads |
| **Parallel** | **three coaches**, each throwing to one batsman, **at the same time** | processes |

* **Concurrent** = many jobs make progress by switching between them. One core is enough.
* **Parallel** = many jobs really run at the same moment. Needs several cores.

##### I/O-bound vs CPU-bound work
**I/O** means *Input / Output*: reading or writing files, downloading from the internet, talking to a database, waiting for the user, `time.sleep()`.

* **I/O-bound** work spends most of its time **waiting**. The CPU has nothing to do while it waits.
  🏏 Waiting for the **third umpire's decision** - nothing happens until it comes.
* **CPU-bound** work spends most of its time **calculating**. The CPU is busy all the time.
  🏏 Calculating the batting average and strike rate of **every player in 20 years of matches**.

| Task | Type | What helps? |
| --- | --- | --- |
| Downloading scorecards of 100 matches from a website | I/O-bound | threads |
| Reading 1000 commentary files | I/O-bound | threads |
| Getting the live score from every ground | I/O-bound | threads |
| Simulating 10 million matches to predict a winner | CPU-bound | processes |
| Analysing every ball of a full season | CPU-bound | processes |
| Resizing 500 player photos | CPU-bound | processes |

<details>
<summary><b>Quick check:</b> Sending a notification to 200 fans every time a wicket falls - I/O-bound or CPU-bound?</summary>
I/O-bound. Each notification goes over the internet, and the program mostly waits for the server to reply.
</details>

##### Measuring time
To check whether our code gets faster, we measure it. `time.perf_counter()` gives a very exact clock value in seconds. We read it **before** and **after** the code - the difference is the time taken.

`time.sleep(seconds)` pauses the program. In this session we use it to **pretend** a slow job, like downloading a file.

In [ ]:
import time

start = time.perf_counter()
time.sleep(1.5)                                     # "wait" 1.5 seconds
end = time.perf_counter()

print(f"That took {end - start:.2f} seconds")       # :.2f -> 2 decimal places

##### Our starting point: one job after another (sequential)
We want to download the scorecards of three matches. Each download takes 1 second (we pretend with `sleep`).

In [ ]:
def download_scorecard(match, seconds=1):
    print(f"Downloading {match} ...\n", end="")    # "\n" + end="" -> explained a bit later
    time.sleep(seconds)                         # waiting for the internet
    print(f"Finished {match}\n", end="")

start = time.perf_counter()

download_scorecard("IND vs AUS")
download_scorecard("ENG vs SA")
download_scorecard("NZ vs PAK")

print(f"Sequential: {time.perf_counter() - start:.2f} seconds")

```
Sequential - each download waits for the one before it:

time (s)    0         1         2         3
IND vs AUS  [=========]
ENG vs SA             [=========]
NZ vs PAK                       [=========]       total ≈ 3 s
```
For almost all of these 3 seconds the computer was **doing nothing** - only waiting. This is exactly where threads help.

### 3. Multithreading Basics

<details>
<summary>What is Multithreading?</summary>

Multithreading means running several threads inside <b>one process</b>. While one thread is waiting (for a download, a file, a timer ...), another thread can work. Python has a built-in module for this called <code>threading</code> - nothing to install.
</details>

##### Your first thread - step by step
Using a thread always takes three steps:

| Step | Code | Meaning |
| --- | --- | --- |
| 1. create | `t = threading.Thread(target=download_scorecard, args=("IND vs AUS",))` | prepare the thread. `target` = **which function** to run (only the name, **no** brackets). `args` = a **tuple** with the function's arguments |
| 2. start | `t.start()` | the thread starts running the function **in the background**. `start()` does not wait |
| 3. wait | `t.join()` | the main thread waits here until `t` is finished |

Let's try it with **one** thread first.

In [ ]:
t = threading.Thread(target=download_scorecard, args=("IND vs AUS",))    # 1. create

print("Thread created. Is it running?", t.is_alive())
t.start()                                                                  # 2. start
print("Thread started. Is it running?", t.is_alive())

t.join()                                                                   # 3. wait
print("After join().   Is it running?", t.is_alive())

<details>
<summary>What happened, step by step?</summary>

1. <code>threading.Thread(...)</code> only <b>prepared</b> the thread. Nothing ran yet, so <code>is_alive()</code> was <code>False</code>.
2. <code>t.start()</code> started the thread. It began running <code>download_scorecard("IND vs AUS")</code> and printed "Downloading ...". At the same time, the main thread went on to the next line and printed "Thread started ... True".
3. <code>t.join()</code> made the main thread <b>wait</b> about 1 second, until the download was finished.
4. After <code>join()</code> the thread had ended, so <code>is_alive()</code> was <code>False</code> again.

With one thread there is no speed-up yet - the main thread just waited. Now let's use three.
</details>

##### Three downloads with three threads

In [ ]:
start = time.perf_counter()

t1 = threading.Thread(target=download_scorecard, args=("IND vs AUS",))
t2 = threading.Thread(target=download_scorecard, args=("ENG vs SA",))
t3 = threading.Thread(target=download_scorecard, args=("NZ vs PAK",))

t1.start()          # all three start almost at the same moment
t2.start()
t3.start()

t1.join()           # wait for all three to finish
t2.join()
t3.join()

print(f"With threads: {time.perf_counter() - start:.2f} seconds")

```
With threads - the waiting happens at the same time:

time (s)    0         1
IND vs AUS  [=========]
ENG vs SA   [=========]
NZ vs PAK   [=========]          total ≈ 1 s
```

<details>
<summary>Why is the order of "Finished ..." different from the order we started them?</summary>
All three threads finish their <code>sleep</code> at almost the same moment, and the operating system decides which one goes first. We cannot control this order, so our code should never depend on it.
</details>

<details>
<summary>Why do we write <code>print(f"...\n", end="")</code> in <code>download_scorecard</code>?</summary>
A normal <code>print()</code> writes the text and the new line in <b>two</b> steps. If several threads print at the same moment, another thread can write in between, and two lines get stuck together, like <code>Downloading A ...Downloading B ...</code>. Putting <code>\n</code> inside the text and using <code>end=""</code> writes the whole line in <b>one</b> step.
</details>

##### What if we forget `join()`?
The main thread does not wait. It goes straight to the next line, so the message below is printed **before** the download is finished.

In [ ]:
t = threading.Thread(target=download_scorecard, args=("WI vs SL", 0.5))
t.start()
print("Main thread: all scorecards downloaded!   <- WRONG, printed too early")

t.join()            # now we really wait
print("Main thread: now the download is really finished")

##### Many threads with a loop
In real programs we keep the threads in a **list**: first start all of them, then join all of them.

In [ ]:
matches = ["IND vs AUS", "ENG vs SA", "NZ vs PAK", "WI vs SL", "BAN vs AFG"]

start = time.perf_counter()

threads = []
for match in matches:
    t = threading.Thread(target=download_scorecard, args=(match,))
    threads.append(t)
    t.start()                   # start each thread

for t in threads:
    t.join()                    # then wait for all of them

print(f"{len(matches)} scorecards in {time.perf_counter() - start:.2f} seconds")

The same thing, shorter, with a **list comprehension**:

In [ ]:
start = time.perf_counter()

threads = [threading.Thread(target=download_scorecard, args=(m, 0.5)) for m in matches]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"{len(matches)} scorecards in {time.perf_counter() - start:.2f} seconds")

##### Common mistakes
| Mistake | What happens | Correct |
| --- | --- | --- |
| `target=download_scorecard("IND vs AUS")` | the brackets **run** the function right away in the main thread - no new thread does the work | `target=download_scorecard, args=("IND vs AUS",)` |
| `args=("IND vs AUS")` - no comma | `("IND vs AUS")` is just a string, not a tuple, so every **letter** becomes a separate argument → `TypeError` | `args=("IND vs AUS",)` - the comma makes it a tuple |
| `start()` and `join()` in the **same** loop | each thread is waited for before the next one starts → sequential again | start all, then join all |
| `t.run()` instead of `t.start()` | runs the function in the **current** thread - no new thread | `t.start()` |
| starting the same thread twice | `RuntimeError: threads can only be started once` | create a new `Thread` |

In [ ]:
print(type(("IND vs AUS")))         # <class 'str'>   - brackets alone do NOT make a tuple
print(type(("IND vs AUS",)))        # <class 'tuple'> - the comma does

t = threading.Thread(target=download_scorecard, args=("ENG vs SA", 0.1))
t.start()
t.join()
try:
    t.start()                       # a thread can only be started once
except RuntimeError as e:
    print("RuntimeError:", e)

##### Naming threads
We can give every thread a name with `name=...`. Inside the function, `threading.current_thread().name` tells us which thread is running.

In [ ]:
def pad_up(seconds):
    me = threading.current_thread()
    print(f"{me.name} is padding up\n", end="")
    time.sleep(seconds)

openers = [threading.Thread(target=pad_up, args=(0.5,), name=name) for name in ["Rohit", "Gill", "Kohli"]]
for t in openers:
    t.start()
for t in openers:
    t.join()
print("All batsmen ready")

##### The GIL (Global Interpreter Lock)

<details>
<summary>What is the GIL?</summary>

The <b>GIL</b> is a rule inside Python: <b>only one thread at a time can run Python code</b>, even if the computer has many cores.

🏏 <b>Cricket example:</b> there is only <b>one ball</b> on the field. Many players (threads) are on the field, but only the player holding the ball can do something with it.

* A thread that is <b>waiting</b> (<code>sleep</code>, download, reading a file) <b>gives the ball away</b>, so other threads can work. → Threads <b>do</b> speed up I/O-bound work.
* A thread that is <b>calculating</b> keeps the ball. The threads only take turns. → Threads <b>do not</b> speed up CPU-bound work.
* Every <b>process</b> has its own Python and its <b>own GIL</b> - like separate matches with separate balls. → Processes <b>do</b> speed up CPU-bound work.
</details>

##### Proof: a calculation with threads
`cpu_heavy(n)` only calculates - it never waits. It is saved in `session_4_workers.py`, because we will use it with processes later too.

In [4]:
from session_4_workers import cpu_heavy
import time ,threading
N = 10_000_000
TASKS = 4

start = time.perf_counter()
for _ in range(TASKS):
    cpu_heavy(N)
sequential_time = time.perf_counter() - start
print(f"Sequential  : {sequential_time:.2f} s")

start = time.perf_counter()
threads = [threading.Thread(target=cpu_heavy, args=(N,)) for _ in range(TASKS)]
for t in threads: t.start()
for t in threads: t.join()
thread_time = time.perf_counter() - start
print(f"4 threads   : {thread_time:.2f} s   <- no real speed-up, because of the GIL")

Sequential  : 3.21 s
4 threads   : 3.31 s   <- no real speed-up, because of the GIL


<details>
<summary><b>Quick check:</b> Threads made the downloads 3 times faster, but not this calculation. Why?</summary>
The downloads were <b>waiting</b> (<code>sleep</code>). Waiting threads give away the GIL, so all the waiting happened at the same time. The calculation never waits, so the threads only took turns and the total work stayed the same.
</details>

##### Race condition - when threads share data

<details>
<summary>What is a Race Condition?</summary>

Threads share memory. When two threads <b>read and change the same variable at the same time</b>, the result can be wrong. This bug is called a <b>race condition</b>.

🏏 <b>Cricket example:</b> two scorers update the same scoreboard. The board shows 100. Scorer A reads 100 and starts writing 101. Before he finishes, scorer B also reads 100 and writes 101. Two runs were scored, but the board shows 101 - <b>one run is lost</b>.

<code>runs = runs + 1</code> looks like one step, but it is really three: <b>read</b> runs → <b>add</b> 1 → <b>write</b> it back. If another thread runs between the read and the write, an update is lost.
</details>

```
 Thread A (scorer A)        Thread B (scorer B)        runs on the board
 read runs  -> 100                                      100
                            read runs  -> 100           100
 write 100 + 1                                          101
                            write 100 + 1               101   <- should be 102!
```

In [5]:
total_runs = 0

def score_runs(balls):
    global total_runs                   # we change a variable that lives outside the function
    for _ in range(balls):
        current = total_runs            # 1. read the board
        time.sleep(0)                   # 2. let other threads run here (makes the bug easy to see)
        total_runs = current + 1        # 3. write - may overwrite another thread's update!

scorers = [threading.Thread(target=score_runs, args=(1000,)) for _ in range(5)]
for t in scorers: t.start()
for t in scorers: t.join()

print("Expected runs:", 5 * 1000)
print("Actual runs  :", total_runs)     # much smaller - runs were lost

Expected runs: 5000
Actual runs  : 1277


<details>
<summary>Why <code>time.sleep(0)</code>?</summary>
<code>sleep(0)</code> means "another thread may run now". Without it the bug is still there, but it happens only rarely - and a bug that shows up once in a thousand runs is very hard to find. Here we make it happen every time so we can see it.
</details>

##### Lock - the fix
A `Lock` lets **only one thread at a time** into a block of code.

🏏 **Cricket example:** there is only **one scorebook pen**. Whoever holds the pen may write. Everyone else waits until the pen is put down.

* `with lock:` - takes the lock at the start of the block and **always** gives it back at the end, even if an error happens (like `with open(...)` always closes a file).
* (`lock.acquire()` / `lock.release()` do the same by hand - `with` is easier and safer.)

In [9]:
total_runs = 0
pen = threading.Lock()

def score_runs_safe(balls):
    global total_runs
    for _ in range(balls):
        with pen:                       # only one thread at a time can be inside this block
            current = total_runs
            time.sleep(0)
            total_runs = current + 1

scorers = [threading.Thread(target=score_runs_safe, args=(1000,)) for _ in range(5)]
for t in scorers: t.start()
for t in scorers: t.join()

print("Expected runs:", 5 * 1000)
print("Actual runs  :", total_runs)     # always correct now

Expected runs: 5000
Actual runs  : 5000


<details>
<summary><b>Quick check:</b> Why not put a lock around <i>all</i> the code in a thread, to be safe?</summary>
Then only one thread could work at a time, and we would be back to sequential speed. Lock only the small part that changes shared data.
</details>

### 4. Multiprocessing Basics

<details>
<summary>What is Multiprocessing?</summary>

Multiprocessing means doing work in several <b>separate processes</b>. Each process has its own Python, its own memory and its <b>own GIL</b>, so calculations really run at the same time on different cores. Python's built-in module for this is <code>multiprocessing</code>. It looks almost the same as <code>threading</code> - we use <code>Process</code> instead of <code>Thread</code>.

🏏 <b>Cricket example:</b> instead of more players in one stadium (threads), we play <b>several matches in different stadiums</b> at the same time. Each match has its own ball (its own GIL), so they don't wait for each other.
</details>

##### Threads vs processes
| | Thread | Process |
| --- | --- | --- |
| Memory | shared with the other threads | its own, separate memory |
| Start-up | fast and light | slower (a new Python must start) |
| Really runs Python code at the same time? | no (GIL) | **yes** |
| Best for | waiting jobs (I/O-bound) | calculation jobs (CPU-bound) |
| If it crashes ... | it can bring down the whole program | the other processes keep running |

##### Your first process - as a normal Python script
How does a new process know what to run? On Windows, Python starts a **new, empty Python** for the process. This new Python **imports your file** to find the function it must run.

That causes a problem: when the file is imported, all its top-level lines run again - including the lines that create new processes. Every new process would create more processes... Python stops this with the error
`RuntimeError: An attempt has been made to start a new process before the current process has finished its bootstrapping phase.`

The fix is the `if __name__ == "__main__":` check:
* when we **run** the file ourselves, `__name__` is `"__main__"` → the block runs and creates the processes,
* when the new process **imports** the file, `__name__` is different → the block is skipped.

**Rule: in every `.py` file that uses multiprocessing, put the code that starts processes under `if __name__ == "__main__":`.**

The cell below **writes a small script to a file** (`%%writefile`) and the next cell runs it.

In [11]:
%%writefile process_demo.py
import multiprocessing
import os
import time

def play_match(match, seconds):
    print(f"{match} started in process {os.getpid()}")
    time.sleep(seconds)
    print(f"{match} finished")

if __name__ == "__main__":                       # <- only the main program runs this block
    print("Main process id:", os.getpid())
    p1 = multiprocessing.Process(target=play_match, args=("IND vs AUS", 1))
    p2 = multiprocessing.Process(target=play_match, args=("ENG vs SA", 1))

    start = time.perf_counter()
    p1.start()
    p2.start()
    p1.join()
    p2.join()
    print(f"Both matches done in {time.perf_counter() - start:.2f} s")

Overwriting process_demo.py


In [ ]:
import sys
!"{sys.executable}" -u process_demo.py

<details>
<summary>What does <code>!"{sys.executable}" -u process_demo.py</code> mean?</summary>
<ul>
<li><code>!</code> runs a terminal command from a notebook cell.</li>
<li><code>{sys.executable}</code> is the Python that runs this notebook, so the script uses the same Python.</li>
<li><code>-u</code> prints every line straight away (without it, lines can show up in a mixed-up order).</li>
</ul>
It is the same as typing <code>python process_demo.py</code> in a terminal.
</details>

##### Multiprocessing inside Jupyter (on Windows)
A notebook is **not** a `.py` file, so a new process **cannot import a function written in a notebook cell**. You get an error like `AttributeError: Can't get attribute 'play_match' on <module '__main__'>`. Two ways around it:
1. write a script, like above, and run it, or
2. keep the worker functions in a normal `.py` file - here **`session_4_workers.py`** - and import them. We do this from now on.

Two more things to know in Jupyter:
* The notebook itself is the main program, so the cells below don't need the `__main__` check. **Your `.py` files always do.**
* A `print()` inside a new process usually does **not** show up in the notebook. So our worker functions send their results back instead of printing them.

Let's look at the worker function we will use:

In [14]:
import inspect
import session_4_workers as workers

print(inspect.getsource(workers.cpu_heavy))        # shows the code of a function

def cpu_heavy(n):
    """CPU-bound work: a pure-Python loop that only calculates (no waiting).
    Think of it as crunching the statistics of n deliveries."""
    total = 0
    for i in range(n):
        total += i * i
    return total



##### Creating processes
`multiprocessing.Process` works just like `threading.Thread`: `target`, `args`, `start()`, `join()`. A process also has:
* `pid` - its process id,
* `exitcode` - `0` means it finished normally.

In [16]:
import multiprocessing
import os
print("Main (notebook) process id:", os.getpid())

p = multiprocessing.Process(target=workers.cpu_heavy, args=(1_000_000,))
p.start()
print("New process id            :", p.pid)        # a different number - it is a separate program
p.join()
print("Exit code                 :", p.exitcode)

Main (notebook) process id: 18716
New process id            : 8460
Exit code                 : 0


##### The real speed-up: calculations with processes
The same `cpu_heavy` job as in the GIL example - now with 4 processes instead of 4 threads.

In [17]:
start = time.perf_counter()
processes = [multiprocessing.Process(target=workers.cpu_heavy, args=(N,)) for _ in range(TASKS)]
for p in processes: p.start()
for p in processes: p.join()
process_time = time.perf_counter() - start

print(f"Sequential  : {sequential_time:.2f} s")
print(f"4 threads   : {thread_time:.2f} s")
print(f"4 processes : {process_time:.2f} s   <- really running at the same time")

Sequential  : 3.21 s
4 threads   : 3.31 s
4 processes : 1.23 s   <- really running at the same time


<details>
<summary>Why isn't it exactly 4 times faster?</summary>

1. Starting a process takes time - a whole new Python must start.
2. The data has to be packed up and sent between the processes.
3. Other programs on your computer also use the cores.

So processes are only worth it when each job is <b>big enough</b>. For tiny jobs, the extra cost makes them <b>slower</b> than a normal loop.
</details>

##### Processes do NOT share memory
Every process has its own copy of every variable. If a process changes a list, it changes **only its own copy**. Let's run the same function `add_to_squad` once in a **thread** and once in a **process**:

In [ ]:
print(inspect.getsource(workers.add_to_squad))

t = threading.Thread(target=workers.add_to_squad, args=("Rohit",))          # a thread shares our memory
t.start()
t.join()
print("After the thread :", workers.squad)      # ['Rohit'] - the thread changed OUR list

p = multiprocessing.Process(target=workers.add_to_squad, args=("Kohli",))   # a process has its own memory
p.start()
p.join()
print("After the process:", workers.squad)      # still ['Rohit'] - "Kohli" was added only to the process's own copy

<details>
<summary>So how do processes send data to each other?</summary>
With special tools such as <code>multiprocessing.Queue</code>, <code>Pipe</code>, <code>Value</code> and <code>Manager</code> (see Part B). But the easiest way is to let each job simply <b>return</b> its result - and that is what Pool Executors do, in the next section.
</details>

### 5. Pool Executors (`concurrent.futures`)

<details>
<summary>What is a Pool Executor?</summary>

Creating, starting and joining every thread or process by hand is a lot of work, and getting results back is hard. The built-in module <code>concurrent.futures</code> makes it easy. It gives us a <b>pool</b> - a fixed team of workers that share all the jobs:
<ul>
<li><code>ThreadPoolExecutor</code> - a team of <b>threads</b> (for waiting jobs)</li>
<li><code>ProcessPoolExecutor</code> - a team of <b>processes</b> (for calculation jobs)</li>
</ul>
Both are used in the same way, so switching is a one-word change. <b>For most real programs, this is what you should use.</b>

🏏 <b>Cricket example:</b> a team of 4 analysts and a pile of 100 match videos. Each analyst takes the next video from the pile as soon as they finish the current one.
</details>

<details>
<summary>What is a <b>Future</b>?</summary>

When we give a job to the pool, we get back a <code>Future</code> - a <b>receipt for a result that is not ready yet</b>.

🏏 <b>Cricket example:</b> you order a samosa at the stadium and get a token. The token is not the samosa, but you can use it later to collect the samosa when it is ready.

| Method | Meaning |
| --- | --- |
| `future.result()` | wait for the result and return it (if the job had an error, the error is raised here) |
| `future.done()` | `True` when the job is finished |
</details>

##### ThreadPoolExecutor with `submit()`
* `executor.submit(function, arg1, arg2, ...)` gives one job to the pool and returns a `Future` straight away.
* `with ThreadPoolExecutor(...) as executor:` - at the end of the `with` block Python waits for all jobs, so we don't need `join()`.
* `max_workers` = how many threads are in the team.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

def get_scorecard(match, seconds):
    time.sleep(seconds)                         # pretend to download
    return f"{match} scorecard ({seconds}s)"    # a normal return value!

with ThreadPoolExecutor(max_workers=3) as executor:
    future = executor.submit(get_scorecard, "IND vs AUS", 1)
    print("done yet?", future.done())
    print("result   :", future.result())        # waits until the result is ready
    print("done yet?", future.done())

##### Many jobs with `submit()` and `as_completed()`
`as_completed(futures)` gives back each future **as soon as it is finished** - the fastest first, not in the order we gave them.

In [ ]:
jobs = {"IND vs AUS (Test)": 1.5, "ENG vs SA (T20)": 0.3, "NZ vs PAK (ODI)": 0.8}

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(get_scorecard, match, sec) for match, sec in jobs.items()]

    for future in as_completed(futures):
        print(f"{time.perf_counter() - start:.1f}s  finished: {future.result()}")

print(f"Total: {time.perf_counter() - start:.2f} s")

##### `executor.map()` - results in the same order as the input
Works like the built-in `map()`. Here we have 10 downloads but only 5 threads, so each thread does two downloads, one after the other.

In [ ]:
matches = [f"Match {i}" for i in range(1, 11)]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(get_scorecard, matches, [0.5] * len(matches)))     # two lists -> two arguments

print(results[:3], "...")
print(f"10 scorecards, 5 threads: {time.perf_counter() - start:.2f} s")    # about 1 s (2 rounds of 0.5 s)

| | `submit()` + `as_completed()` | `map()` |
| --- | --- | --- |
| Order of results | as soon as each job finishes | same order as the input |
| Gives back | `Future` objects | the results themselves |
| Good for | showing progress, handling each error separately | simple "do this for every item" |

##### Errors inside jobs
If a job raises an error, your program does **not** stop. The error is kept inside the `Future` and raised when you call `future.result()` - so put that call inside `try / except`.

In [ ]:
with ThreadPoolExecutor() as executor:
    future_to_match = {executor.submit(workers.may_fail, n): n for n in range(1, 6)}    # {future: match number}

    for future in as_completed(future_to_match):
        match_no = future_to_match[future]
        try:
            print(f"match {match_no}: {future.result()}")
        except ValueError as e:
            print(f"match {match_no}: FAILED - {e}")

##### ProcessPoolExecutor - same code, real parallel work
To use processes, just change the class name. Remember: the function must be in a `.py` file that can be imported (here `session_4_workers.py`).

In [ ]:
with ProcessPoolExecutor(max_workers=4) as executor:
    print(list(executor.map(workers.square, [1, 2, 3, 4, 5])))      # square() runs in other processes

In [ ]:
for Executor in [ThreadPoolExecutor, ProcessPoolExecutor]:
    start = time.perf_counter()
    with Executor(max_workers=4) as executor:
        list(executor.map(workers.cpu_heavy, [N] * TASKS))
    print(f"{Executor.__name__:<20}: {time.perf_counter() - start:.2f} s")

<details>
<summary>How many workers should I use?</summary>

* <b>Calculation jobs</b> (processes): about the number of cores - <code>os.cpu_count()</code>. More than that does not help.
* <b>Waiting jobs</b> (threads): often more than the number of cores (10, 20, 50 ...), because most of them are just waiting. If you don't give <code>max_workers</code>, Python picks a sensible number for you.
* Be polite to websites - 1000 threads hitting one website at the same time looks like an attack. Keep <code>max_workers</code> small.
</details>

##### Which one should I use?
```
Is the job mostly WAITING (internet, files, sleep)?
├── Yes -> ThreadPoolExecutor
└── No, it is CALCULATING -> ProcessPoolExecutor
```

| | Threads | Processes |
| --- | --- | --- |
| Cricket example | many players, one stadium | many matches, many stadiums |
| Best for | waiting jobs (I/O-bound) | calculation jobs (CPU-bound) |
| Memory | shared | separate |
| Start-up cost | small | bigger |
| Watch out for | race conditions → use a `Lock` | the `__main__` check, functions must be in a `.py` file |

<details>
<summary><b>Quick check:</b> Which one would you use?</summary>

| Job | Answer |
| --- | --- |
| Download 300 player photos from a website | `ThreadPoolExecutor` - waiting for the internet |
| Simulate 50 million balls to estimate a bowler's economy | `ProcessPoolExecutor` - pure calculation |
| Count the words in 2000 commentary files | threads - mostly reading files |
| Two threads add runs to the same total | protect the total with a `Lock` |
</details>

##### Good habits
1. **Measure first** with `time.perf_counter()` - is the code really slow? Is it waiting or calculating?
2. **Use Pool Executors** - they are easier and safer than creating threads and processes by hand.
3. **Share as little as possible** - give each job its data and let it `return` the result.
4. If threads must change the same data, protect it with **`with lock:`**.
5. Always **`join()`** threads and processes, or use `with` for executors.
6. In `.py` files, put multiprocessing code under **`if __name__ == "__main__":`**.
7. **Handle errors** - call `future.result()` inside `try / except`.

##### Glossary
| Word | Meaning |
| --- | --- |
| CPU / core | the part of the computer that runs instructions; one core does one job at a time |
| Process | a running program with its own memory and PID |
| Thread | a worker inside a process; threads share the process's memory |
| Main thread | the thread every program starts with |
| Concurrent | many jobs make progress by switching between them |
| Parallel | many jobs really run at the same moment on different cores |
| I/O-bound | the job mostly waits (internet, files, sleep) |
| CPU-bound | the job mostly calculates |
| GIL | the rule that only one thread at a time runs Python code in a process |
| Race condition | a bug where threads change the same data at the same time and the result is wrong |
| Lock | lets only one thread at a time into a block of code |
| Pool / Executor | a fixed team of workers that share many jobs |
| Future | a receipt for a result that is not ready yet |

---
# Part B - Advanced (optional)
These topics are useful in bigger programs. Read them once Part A is clear.

### 6. More About Threads

##### A thread as a class
We can make our own class that **inherits** from `threading.Thread` and write the work in a method called `run()`. `start()` calls `run()` for us in the new thread.

In [ ]:
import random

class Innings(threading.Thread):
    def __init__(self, batsman, balls):
        super().__init__(name=f"Innings-{batsman}")    # set up the Thread part
        self.batsman = batsman
        self.balls = balls
        self.runs = 0                                   # the thread keeps its result here

    def run(self):                                      # this runs in the new thread
        for _ in range(self.balls):
            time.sleep(0.05)                            # each ball takes a little time
            self.runs += random.choice([0, 1, 1, 2, 4, 6])

random.seed(10)
innings = [Innings("Kohli", 20), Innings("Root", 20), Innings("Williamson", 20)]
for i in innings:
    i.start()
for i in innings:
    i.join()

for i in innings:
    print(f"{i.name:<18} {i.runs} runs off {i.balls} balls")

##### Getting results back from threads
A thread does **not** give back the return value of its function. Simple ways to get results:
1. pass a **dict** to the threads, and let each thread write **its own key** (below),
2. keep the result on the object (like `self.runs` above),
3. use a `queue.Queue` (section 7) - or just use a `ThreadPoolExecutor` (Part A), which is the easiest.

In [ ]:
def fetch_live_score(team, results):
    time.sleep(random.uniform(0.2, 0.6))        # pretend to call a live-score website
    results[team] = random.randint(150, 350)    # each thread writes its own key

results = {}
teams = ["India", "Australia", "England", "South Africa"]
threads = [threading.Thread(target=fetch_live_score, args=(team, results)) for team in teams]

for t in threads: t.start()
for t in threads: t.join()

print(results)          # the order depends on which thread finished first

##### Keyword arguments: `kwargs` and other thread information
`args` is a tuple of normal arguments; `kwargs` is a **dict** of keyword arguments.

| Function | Meaning |
| --- | --- |
| `threading.main_thread()` | the main thread |
| `threading.active_count()` | how many threads are running now |
| `threading.get_ident()` | a number that identifies the current thread |

In [ ]:
t = threading.Thread(target=download_scorecard, args=("AUS vs ENG",), kwargs={"seconds": 0.3})
t.start()
print("Main thread   :", threading.main_thread().name)
print("Active threads:", threading.active_count())      # Jupyter also runs a few threads of its own
print("This thread id:", threading.get_ident())
t.join()

##### Daemon threads
Normally Python **waits** for all threads to finish before the program ends. A **daemon thread** is a background helper that Python does **not** wait for. When the main program ends, daemon threads are stopped at once - even in the middle of their work.

🏏 **Cricket example:** the stadium music. When the match is over, nobody waits for the song to finish - the music just stops.

Use daemon threads for background helpers (like auto-save). Never use them for work that must finish, like writing a file. Create one with `threading.Thread(target=..., daemon=True)`.

In Jupyter the notebook keeps running after a cell ends, so it is hard to see the effect. The cell below writes a small script and the next cell runs it.

In [ ]:
%%writefile daemon_demo.py
import threading
import time

def background_task(label):
    time.sleep(1)
    print(f"{label} thread finished its work")

normal = threading.Thread(target=background_task, args=("Normal",))
daemon = threading.Thread(target=background_task, args=("Daemon",), daemon=True)
daemon.start()

print("Main program ends now. Is it a daemon?", daemon.daemon)
# The daemon thread is stopped when the program ends, so its message never appears.
# Try it: also start `normal` and see that Python waits for that one.

In [ ]:
!"{sys.executable}" daemon_demo.py

##### Timer - run a function after a delay
`threading.Timer(seconds, function, args)` waits and then calls the function. `cancel()` stops it if it has not run yet.

In [ ]:
def announce(msg):
    print("Announcement:", msg)

timer = threading.Timer(1.0, announce, args=("Drinks break!",))
timer.start()
print("Timer started, waiting 1 second ...")
timer.join()

rain_timer = threading.Timer(5.0, announce, args=("This never prints",))
rain_timer.start()
rain_timer.cancel()                         # cancelled before it ran
print("Second timer cancelled")

### 7. More Synchronization Tools

##### A lock inside a class - a safe scoreboard
A team can lose at most 10 wickets. The **check** ("are wickets left?") and the **change** must happen together, inside the lock - otherwise two threads could both pass the check at the same time.

In [ ]:
class AllOutError(Exception):
    pass

class Scoreboard:
    def __init__(self, team):
        self.team = team
        self.__runs = 0
        self.__wickets = 0
        self._lock = threading.Lock()

    def add_runs(self, runs):
        with self._lock:
            self.__runs += runs

    def take_wicket(self):
        with self._lock:                            # check and change together
            if self.__wickets >= 10:
                raise AllOutError(f"{self.team} are already all out")
            time.sleep(0.001)                       # pretend updating the board is slow
            self.__wickets += 1

    def __str__(self):
        return f"{self.team}: {self.__runs}/{self.__wickets}"

board = Scoreboard("India")
refused = []

def bowler_appeals():
    try:
        board.take_wicket()
    except AllOutError as e:
        refused.append(str(e))

bowlers = [threading.Thread(target=bowler_appeals) for _ in range(12)]     # 12 wickets, but only 10 allowed
for t in bowlers: t.start()
for t in bowlers: t.join()
board.add_runs(287)

print(board)                    # never more than 10 wickets
print("Refused:", refused)

<details>
<summary>What is a <b>Deadlock</b>?</summary>

A deadlock is when threads wait for each other <b>forever</b>. For example: thread A holds lock 1 and waits for lock 2, while thread B holds lock 2 and waits for lock 1. Neither can go on - the program just hangs.

🏏 <b>Cricket example:</b> both batsmen are stuck in the middle of the pitch, each waiting for the other to go back first. Nobody moves.

How to avoid it:
1. Always take several locks in the <b>same order</b> in every thread.
2. Keep the code inside a lock short.
3. Use a timeout: <code>lock.acquire(timeout=2)</code> gives up after 2 seconds and returns <code>False</code>.
</details>

##### RLock (re-entrant lock)
A normal `Lock` cannot be taken twice - not even by the thread that already holds it. It would wait for itself forever. An `RLock` remembers **which thread** holds it, so the same thread may take it again. This is needed when one locked method calls another locked method.

In [ ]:
lock = threading.Lock()
lock.acquire()
print("Second acquire of a Lock worked?", lock.acquire(timeout=1))   # False - without the timeout it would wait forever
lock.release()

class Scorebook:
    def __init__(self):
        self.runs = {}
        self._lock = threading.RLock()          # try threading.Lock() here -> the cell hangs!

    def add_ball(self, batsman, runs):
        with self._lock:
            self.runs[batsman] = self.runs.get(batsman, 0) + runs

    def add_over(self, balls):
        with self._lock:                        # holds the lock ...
            for batsman, runs in balls:
                self.add_ball(batsman, runs)    # ... and add_ball() takes it again

book = Scorebook()
book.add_over([("Kohli", 4), ("Kohli", 1), ("Gill", 6), ("Gill", 0), ("Gill", 1), ("Kohli", 2)])
print(book.runs)

##### Semaphore
A `Semaphore(n)` is like a lock that lets **up to `n` threads** in at the same time. Use it when a resource is limited (for example: at most 3 downloads from one website at once).

🏏 **Cricket example:** there are only **2 practice nets**. Five batsmen want to practise, so only two can bat at a time. The others wait until a net is free.

In [ ]:
nets = threading.Semaphore(2)

def practise(batsman):
    with nets:                                       # waits if both nets are taken
        print(f"{time.perf_counter() - t0:4.1f}s  {batsman} enters a net\n", end="")
        time.sleep(1)
        print(f"{time.perf_counter() - t0:4.1f}s  {batsman} leaves\n", end="")

t0 = time.perf_counter()
batsmen = ["Rohit", "Gill", "Kohli", "Iyer", "Pant"]
threads = [threading.Thread(target=practise, args=(b,)) for b in batsmen]
for t in threads: t.start()
for t in threads: t.join()

<details>
<summary>Semaphore vs BoundedSemaphore</summary>
With a normal <code>Semaphore</code> you can call <code>release()</code> more often than <code>acquire()</code> by mistake, which quietly raises the limit. A <code>BoundedSemaphore</code> raises a <code>ValueError</code> instead, so it is the safer choice.
</details>

##### Event
An `Event` is a simple **signal** between threads - a flag that is `False` or `True`.
* `event.wait()` - wait until the flag becomes `True`
* `event.set()` - make it `True` and wake up **all** waiting threads
* `event.clear()` - make it `False` again; `event.is_set()` - check it

🏏 **Cricket example:** all fielders wait in position until the umpire calls **"Play!"**.

In [ ]:
play = threading.Event()

def fielder(name):
    print(f"{name} is waiting in position\n", end="")
    play.wait()                             # waits here until the event is set
    print(f"{name} is ready!\n", end="")

fielders = ["Jadeja", "Kohli", "Gill"]
threads = [threading.Thread(target=fielder, args=(f,)) for f in fielders]
for t in threads: t.start()

time.sleep(1)
print("Umpire: PLAY!")
play.set()                                  # wakes up every waiting thread at once

for t in threads: t.join()

##### Condition
A `Condition` lets threads wait until **something has changed** in the shared data. Another thread changes the data and then wakes them up.
* `cond.wait_for(function)` - wait until woken up **and** the function returns `True`
* `cond.notify_all()` - wake up all waiting threads

🏏 **Cricket example:** the players wait until the groundsman says the **covers are off** after the rain.

In [ ]:
covers_off = False
cond = threading.Condition()

def player(name):
    with cond:
        cond.wait_for(lambda: covers_off)
        print(f"{name} walks out to the field")

def groundsman():
    global covers_off
    time.sleep(1)
    with cond:
        covers_off = True
        print("Groundsman: covers are off!")
        cond.notify_all()

threads = [threading.Thread(target=player, args=(p,)) for p in ["Bumrah", "Siraj", "Jadeja"]]
threads.append(threading.Thread(target=groundsman))
for t in threads: t.start()
for t in threads: t.join()

##### Queue - a safe way for threads to share data
A `queue.Queue` is a **safe line of items** for threads: first in, first out. The locking is built in, so we don't need our own `Lock`.

🏏 **Cricket example:** the batting order - whoever is padded up first goes in first.

| Method | Meaning |
| --- | --- |
| `q.put(item)` | add an item at the end |
| `q.get()` | take the first item out (waits if the queue is empty) |
| `q.task_done()` | say "I have finished the item I took" |
| `q.join()` | wait until every item is finished |
| `q.qsize()` | how many items are waiting |

In [ ]:
import queue

batting_order = queue.Queue()
for player_name in ["Rohit", "Gill", "Kohli"]:
    batting_order.put(player_name)

print("Waiting to bat:", batting_order.qsize())
print("Next in:", batting_order.get())        # first in, first out
print("Next in:", batting_order.get())
print("Waiting to bat:", batting_order.qsize())

##### Producer - Consumer
* **Producers** make work and `put()` it in the queue.
* **Consumers** `get()` work from the queue and handle it.
* To tell a consumer "there is no more work", we send a special value called a **sentinel** (here `None`).

🏏 The **bowler** (producer) bowls deliveries. Two **scorers** (consumers) take each delivery from the queue and count runs and boundaries.

```
 bowler ── put() ──►  [ 4 | 0 | 1 | 6 | ... ]  ── get() ──► scorer 1
                                                ── get() ──► scorer 2
```

In [ ]:
deliveries = queue.Queue()

def bowler(balls):
    for ball in range(1, balls + 1):
        runs = random.choice([0, 0, 1, 1, 2, 4, 6])
        deliveries.put((ball, runs))
        time.sleep(0.05)
    print("Bowler: overs finished")

def scorer(name, tally):
    while True:
        item = deliveries.get()
        if item is None:                         # sentinel -> stop
            deliveries.task_done()
            break
        ball, runs = item
        tally["runs"] += runs
        if runs in (4, 6):
            tally["boundaries"] += 1
        deliveries.task_done()

random.seed(7)
tallies = {"Scorer-1": {"runs": 0, "boundaries": 0}, "Scorer-2": {"runs": 0, "boundaries": 0}}   # each scorer has its own dict

producer = threading.Thread(target=bowler, args=(12,))                 # 2 overs
consumers = [threading.Thread(target=scorer, args=(name, tallies[name])) for name in tallies]

producer.start()
for c in consumers: c.start()

producer.join()                   # wait until every ball is bowled
for _ in consumers:
    deliveries.put(None)          # one sentinel for each consumer
deliveries.join()                 # wait until every ball is handled
for c in consumers: c.join()

print(tallies)
print("Total runs:", sum(t["runs"] for t in tallies.values()))

##### Synchronization tools - summary
| Tool | Cricket example | Use it when |
| --- | --- | --- |
| `Lock` | one scorebook pen | only one thread at a time may change shared data |
| `RLock` | - | the same thread needs to take the lock again |
| `Semaphore(n)` | only `n` practice nets | at most `n` threads may use something at once |
| `Event` | umpire calls "Play!" | one thread tells many waiting threads to go |
| `Condition` | waiting until the covers are off | threads wait until shared data changes |
| `queue.Queue` | batting order | passing work or results between threads |

### 8. More About Multiprocessing

##### How a new process is started
| Start method | How it works | Default on |
| --- | --- | --- |
| `spawn` | starts a **new, empty Python**, which imports your file | **Windows**, macOS |
| `fork` | makes a copy of the running program | older Linux versions |
| `forkserver` | a small helper program makes the copies | Linux (Python 3.14+) |

This is why the `__main__` check is needed on Windows: with `spawn`, every new process imports your file.

##### Ways for processes to share data
| Tool | What it is |
| --- | --- |
| `multiprocessing.Queue` | a queue that works between processes |
| `Value` / `Array` | one number / a list of numbers in **shared memory** |
| `Pool` | the older version of `ProcessPoolExecutor` |
| `Pipe` | a two-way line between exactly **two** processes |
| `Manager` | holds normal Python objects (`dict`, `list` ...) for all processes |

Everything sent between processes is **pickled** (packed into bytes). Numbers, strings, lists, dicts and tuples are fine - **lambdas** and open files are not.

##### multiprocessing.Queue
Works like `queue.Queue`, but between processes.

In [ ]:
print(inspect.getsource(workers.report_pid))

q = multiprocessing.Queue()
processes = [multiprocessing.Process(target=workers.report_pid, args=(f"Stadium-{i}", q)) for i in range(1, 4)]
for p in processes: p.start()
for p in processes: p.join()

for _ in processes:
    print(q.get())          # every process has a different id, but the same parent (the notebook)

A new process sends the scores of an innings one by one; the notebook receives them.

In [ ]:
q = multiprocessing.Queue()
scores = [45, 12, 88, 0, 67]

p = multiprocessing.Process(target=workers.send_scores, args=(q, scores))
p.start()

received = []
while True:
    runs = q.get()                  # waits until the other process puts something
    if runs is None:                # sentinel -> it is done
        break
    received.append(runs)
p.join()

print("Received:", received)
print("Highest score:", max(received))

<details>
<summary><code>queue.Queue</code> vs <code>multiprocessing.Queue</code></summary>
<code>queue.Queue</code> only works between <b>threads</b> of one process. <code>multiprocessing.Queue</code> works between <b>processes</b>. Tip: take the items out of a <code>multiprocessing.Queue</code> <b>before</b> calling <code>join()</code> on the process that sends them - otherwise both sides can end up waiting for each other.
</details>

##### Value and Array - shared memory
* `Value(type, value)` - one shared number. `"i"` = int, `"d"` = float.
* `Array(type, list)` - a fixed-size list of numbers.

They have their own lock (`get_lock()`). `total.value += 1` is still read + write, so **processes can have race conditions too**.

In [ ]:
from multiprocessing import Value, Array

print(inspect.getsource(workers.add_runs_unsafe))
print(inspect.getsource(workers.add_runs_safe))

for worker in [workers.add_runs_unsafe, workers.add_runs_safe]:
    total = Value("i", 0)
    processes = [multiprocessing.Process(target=worker, args=(total, 50_000)) for _ in range(4)]
    for p in processes: p.start()
    for p in processes: p.join()
    print(f"{worker.__name__:<16} expected {4 * 50_000}, got {total.value}")

In [ ]:
over_runs = Array("i", [4, 0, 1, 6, 2, 1])      # runs off each ball of an over

p = multiprocessing.Process(target=workers.double_runs, args=(over_runs,))
p.start()
p.join()

print(list(over_runs))      # changed by the other process - the memory really is shared

##### Pool
`multiprocessing.Pool` is the older way to have a team of worker processes. You will see it in a lot of existing code.

| Method | What it does |
| --- | --- |
| `pool.map(func, items)` | like `map()`, in parallel; results in order |
| `pool.starmap(func, [(a, b), ...])` | like `map`, but each tuple is split into several arguments |
| `pool.apply_async(func, args)` | give one job without waiting; `.get()` returns the result |
| `pool.imap(func, items)` | like `map`, but gives the results one by one |

In [ ]:
from multiprocessing import Pool

with Pool(processes=4) as pool:
    print("map     :", pool.map(workers.square, range(1, 11)))                       # square(1), square(2), ...
    print("starmap :", pool.starmap(workers.power, [(2, 3), (3, 2), (10, 3)]))        # power(2, 3), power(3, 2), ...

    job = pool.apply_async(workers.power, (2, 10))
    print("apply_async:", job.get(timeout=10))

    for value in pool.imap(workers.square, [5, 6, 7]):
        print("imap gave", value)

    pairs = pool.map(workers.which_process, range(8), chunksize=1)                   # chunksize=1 -> one job at a time
    print("process ids used:", {pid for _, pid in pairs})

<details>
<summary>Can I use a lambda with <code>Pool</code> or <code>ProcessPoolExecutor</code>?</summary>
No. The function must be <b>pickled</b> and sent to the other processes, and a lambda cannot be pickled. Use a normal function written at the top level of a <code>.py</code> file.
</details>

In [ ]:
try:
    with Pool(2) as pool:
        pool.map(lambda x: x * 2, [1, 2, 3])
except Exception as e:
    print(type(e).__name__, "->", e)

##### Pipe
`Pipe()` gives **two ends** of a line. Whatever is `send()`-ed into one end can be `recv()`-ed at the other end, in both directions. It connects exactly two processes.

🏏 The field umpire (the notebook) sends review requests to the **third umpire** (another process) and gets a decision back.

In [ ]:
field_umpire, third_umpire_end = multiprocessing.Pipe()

p = multiprocessing.Process(target=workers.third_umpire, args=(third_umpire_end,))
p.start()

for request in ["lbw", "run out", "stumping", "close"]:
    field_umpire.send(request)
    print(f"review {request!r:11} -> {field_umpire.recv()}")

p.join()

##### Manager - share normal Python objects
`multiprocessing.Manager()` starts a small helper process that keeps real Python objects (`dict()`, `list()` ...). All the other processes can use them. Slower than `Value` / `Array`, but works with almost any object.

In [ ]:
with multiprocessing.Manager() as manager:
    scores = manager.dict()
    batting_order = manager.list()

    innings = [("Rohit", 52), ("Gill", 104), ("Kohli", 77)]
    processes = [multiprocessing.Process(target=workers.record_innings, args=(scores, batting_order, name, runs))
                 for name, runs in innings]
    for p in processes: p.start()
    for p in processes: p.join()

    print("scores       :", dict(scores))           # copy to a normal dict before the manager closes
    print("batting order:", list(batting_order))   # the order depends on which process ran first

##### terminate() - stopping a process by force
`terminate()` kills a process at once, like ending it in the Task Manager. Use it only as a last resort: files can be left half-written and clean-up code does not run.

In [ ]:
p = multiprocessing.Process(target=time.sleep, args=(30,))     # a process that would run for 30 seconds
p.start()
print("alive?", p.is_alive(), "pid:", p.pid)

p.terminate()
p.join()
print("alive?", p.is_alive(), "exitcode:", p.exitcode)        # not 0 -> it did not finish normally

### 9. The Future of the GIL and asyncio

<details>
<summary>Is the GIL going away?</summary>
Python 3.13 added an <b>experimental</b> version of Python without the GIL (often installed as <code>python3.13t</code>). From Python 3.14 this "free-threaded" version is officially supported, but it is still a separate, optional download - the normal <code>python</code> you install still has the GIL. So for now the rule stays: <b>threads for waiting jobs, processes for calculation jobs</b>.
</details>

<details>
<summary>What is <code>asyncio</code>?</summary>

<code>asyncio</code> is a third way to do many waiting jobs together - using <b>only one thread</b>. A function written with <code>async def</code> can pause at every <code>await</code> and say "I'm waiting - do something else meanwhile".

🏏 <b>Cricket example:</b> one commentator covering three matches on three screens. Whenever one match pauses (a drinks break, a review), he talks about another.

It can handle thousands of waiting jobs, but it only works with libraries made for asyncio, and it does not help calculation jobs.
</details>

In [ ]:
import asyncio

async def fetch_score(match, seconds):
    print(f"start {match}")
    await asyncio.sleep(seconds)             # waiting - other jobs can run meanwhile
    print(f"done  {match}")
    return match

async def main():
    return await asyncio.gather(             # run all three together
        fetch_score("IND vs AUS", 1),
        fetch_score("ENG vs SA", 1),
        fetch_score("NZ vs PAK", 1),
    )

start = time.perf_counter()
results = await main()                       # in Jupyter we can use await directly
print(results, f"in {time.perf_counter() - start:.2f} s")
# In a normal .py file write:   results = asyncio.run(main())

| | Threads | Processes | asyncio |
| --- | --- | --- | --- |
| Cricket example | many players, one stadium | many matches, many stadiums | one commentator, many matches |
| Runs on | many threads, one process | many processes | one thread |
| Python code really at the same time? | no | **yes** | no |
| Best for | waiting jobs | calculation jobs | very many waiting jobs |

---
## Practice Questions
Six questions from **easy to hard** and a **mini project**. Write your own solution in the code cell under each question.

> **Reminder:** for questions that use **processes**, put your worker functions in a `.py` file (make it with `%%writefile` or your editor) and import it - or write the whole solution as a script with the `if __name__ == "__main__":` check and run it with `!python your_script.py`.

| # | Level | Main topics | Also uses |
| --- | --- | --- | --- |
| 1 | Easy | `Thread`, `start()`, `join()`, thread names | functions, loops, f-strings |
| 2 | Easy-Medium | threads vs sequential timing | list / dict comprehensions, `random` |
| 3 | Medium | race condition, `Lock` | classes, private attributes, `@classmethod`, custom exceptions |
| 4 | Medium | `ThreadPoolExecutor`, `as_completed()`, errors in futures | files, `os` / `pathlib`, JSON, `datetime` |
| 5 | Medium-Hard | `ProcessPoolExecutor` vs `ThreadPoolExecutor`, CPU-bound work | your own module, `__main__` check, `reduce` + `lambda` |
| 6 | Hard (uses Part B) | `queue.Queue`, producer-consumer, `Event` | generators, regular expressions |
| Project | Hardest | threads + processes + locks together | everything you have learned so far |

### Question 1 (Easy) - Bowling an over
Write a function `bowl_over(bowler, balls)` that prints `"<bowler>: ball <n>"` for every ball from `1` to `balls`, waits `0.2` seconds between balls, and at the end prints `"<bowler>: over complete!"`.

a) Create **three** threads for `("Bumrah", 6)`, `("Starc", 6)` and `("Rashid", 4)` using a **loop over a list of tuples**. Give each thread a `name` like `"T-Bumrah"` and print `threading.current_thread().name` inside the function.
b) Join all threads and print `"All overs finished"` only after every thread has ended.
c) What happens to the output and the total time if you call `join()` right after `start()` inside the same loop? Answer in a comment.

```
Bumrah: ball 1   (thread T-Bumrah)
Starc: ball 1    (thread T-Starc)
Rashid: ball 1   (thread T-Rashid)
Bumrah: ball 2   (thread T-Bumrah)
...
All overs finished
```

In [ ]:
# Question 1

### Question 2 (Easy-Medium) - Live scores from 10 grounds
Ten grounds are playing at the same time. Getting the score from one ground is pretended with `time.sleep(random.uniform(0.3, 1.0))` and returns a score `random.randint(120, 350)`.

```python
grounds = ["Wankhede", "Eden Gardens", "Chinnaswamy", "Chepauk", "Lord's",
           "MCG", "SCG", "The Oval", "Newlands", "Gaddafi"]
```
1. Get all scores **one after another** and measure the time with `time.perf_counter()`.
2. Get them again with **one thread per ground**. Each thread writes its result into a shared dict `{ground: score}`.
3. Using **comprehensions only**, print the grounds with a score above 250, and a dict `{"high": [...], "low": [...]}` (high = above 250).
4. Print both times and the speed-up (`sequential / threaded`, rounded to 1 decimal).

```
Sequential: 6.71 s
Threaded  : 0.98 s
Speed-up  : 6.8x
High-scoring grounds: ['Wankhede', 'MCG', ...]
```

In [ ]:
# Question 2

### Question 3 (Medium) - Stadium ticket booking without overbooking
Create a class `Match` for a match with a limited number of tickets.
* A **private** attribute `__tickets_left` and a `Lock`.
* A method `book(fan, n)` that books `n` tickets. If there are not enough tickets left, raise a **custom exception** `HouseFullError` with a clear message. Put `time.sleep(0.001)` between checking and updating, so the race is easy to see.
* A **class variable** `total_bookings` that counts successful bookings for all matches, and a **class method** `bookings()` that returns it.
* `__str__` returns `"<teams>: <tickets_left> tickets left"`.

Then:
1. Create `Match("IND vs PAK", 50)` and start **20 threads**. Each one tries to book `random.randint(1, 5)` tickets for a fan (`"Fan-1"`, `"Fan-2"` ...). Catch `HouseFullError` in the thread function and collect the unlucky fans in a list.
2. Check that tickets left is never negative and that `booked + tickets_left == 50`.
3. Remove the lock, run it a few times and describe what goes wrong.

In [ ]:
# Question 3

### Question 4 (Medium) - Match-report analyser with a ThreadPoolExecutor
1. Create a folder `match_reports` (use `os` or `pathlib`, don't hard-code the full path) and write **8 text files** `report_1.txt` ... `report_8.txt`. Each file contains a random number (50-500) of random words from this list:
   `["boundary", "wicket", "six", "four", "dot", "catch", "lbw", "bowled", "single", "review", "no-ball", "wide"]`
2. Write `analyse(path)` that returns a dict `{"file": name, "words": count, "unique": number_of_unique_words, "top": the_3_most_common_words}`. Add `time.sleep(0.3)` to pretend the disk is slow. If the file does not exist, it should raise `FileNotFoundError`.
3. Add two file paths that **don't exist** to the list and analyse all 10 files with `ThreadPoolExecutor(max_workers=4)` and `submit()`.
4. Use `as_completed()` to print each result as soon as it is ready. Catch the errors from `future.result()` and collect them as `{"file": ..., "error": ...}`.
5. Save a `summary.json` with: the date and time of the analysis (`datetime.now().strftime(...)`), the total time taken, the results sorted by number of words (`sorted` + `lambda`) and the list of errors.
6. Clean up: delete the report files and the folder at the end.

In [ ]:
# Question 4

### Question 5 (Medium-Hard) - Counting primes in parallel
(Counting prime numbers is a classic calculation job - the computer never waits.)
1. Create your own module `prime_tools.py` (use `%%writefile`) with:
    * `is_prime(n)` - `True` if `n` is a prime number,
    * `count_primes(start, end)` - how many primes there are in `range(start, end)`.
2. Split the range `1 .. 2_000_000` into **8 parts** of `(start, end)` tuples - make the list with a comprehension.
3. Count the primes:
    * a) one part after another, with a normal loop,
    * b) with `ProcessPoolExecutor` (`submit()` or `map()`),
    * c) with `ThreadPoolExecutor`.
4. Add up the results with `reduce()` and a `lambda` (not `sum()`), check that all three answers are the same (`148933`) and print the three times. Explain in a markdown cell why (c) is not faster than (a).
5. Bonus: write the whole program as `primes_main.py` with the `if __name__ == "__main__":` check and run it with `!python primes_main.py`. What error do you get if you remove the check?

In [ ]:
# Question 5

### Question 6 (Hard, uses Part B) - Ball-by-ball commentary pipeline
```python
commentary = [
    "Over 1.1 | Bumrah to Warner | FOUR",
    "Over 1.2 | Bumrah to Warner | DOT",
    "Over 1.3 | Bumrah to Warner | WICKET",
    "corrupted line ###",
    "Over 1.4 | Bumrah to Smith | SIX",
    "Over 1.5 | Bumrah to Smith | 1 RUN",
    "Over 1.6 | Bumrah to Labuschagne | WICKET",
]
```
1. Write a **generator** `read_commentary(lines)` that gives one line at a time (sleep `0.1` s per line to pretend it is a live feed).
2. A **producer thread** takes lines from the generator and puts them into a `queue.Queue`.
3. **Two consumer threads** read each line with a **regular expression** using named groups `(?P<over>...)`, `(?P<bowler>...)`, `(?P<batter>...)`, `(?P<result>...)`. The over must look like `1.1` to `1.6` (ball 1 to 6). Good lines go into a `results` list (protect it with a `Lock`), bad lines into an `invalid` list.
4. Stop the consumers with **sentinels**. Use an `Event` called `wicket_fallen` that is set as soon as a `WICKET` is found, and let the main thread print `"Breaking news: a wicket has fallen!"` when it is set (`wicket_fallen.wait(timeout=5)`).
5. At the end, print how often each result happened (as a dict) and the list of batters who got out.

```
Breaking news: a wicket has fallen!
{'FOUR': 1, 'DOT': 1, 'WICKET': 2, 'SIX': 1, '1 RUN': 1}
Out: ['Warner', 'Labuschagne']
Invalid lines: ['corrupted line ###']
```

In [ ]:
# Question 6

---
### Mini Project: Cricket Tournament Data Processor
Build a program that **downloads**, **checks**, **analyses** and **reports** ball-by-ball data for a cricket tournament. It uses classes, comprehensions, generators, files, CSV / JSON, error handling, modules, regular expressions, `datetime`, `random`, `logging`, threads, processes and locks.

**Project structure**
```
tournament_processor/
+-- models.py        # classes Player and Delivery, custom exceptions
+-- data_gen.py      # creates the fake CSV files on the "server"
+-- analysis.py      # the heavy statistics (used by the process pool)
+-- main.py          # runs everything, uses if __name__ == "__main__":
```

**Step 1 - Make the data (`data_gen.py`)**
* Use `random.seed(...)` so the data is the same every time.
* Create a folder `server_data` with one CSV per batsman (`kohli.csv`, `root.csv`, `smith.csv` ... at least 8 players) using the `csv` module. Each row is one ball faced: `match,over,runs,dismissal`, e.g. `3,12.4,4,` or `3,15.2,0,caught` - about **5,000 balls** per file.
* Put some **bad rows** in every file: an impossible over (`12.7`), runs that are not allowed (`5`, `-1`, `abc`), a missing value, an unknown dismissal (`hit by bat`).

**Step 2 - "Download" the files with threads (waiting job)**
* `download(file_name)` copies the file from `server_data` to `downloads` (`shutil`) after `time.sleep(random.uniform(0.2, 0.8))` to pretend it is the internet.
* Use `ThreadPoolExecutor(max_workers=3)` to download all files.
* Keep a shared counter `downloaded_bytes` protected by a `Lock`, and print a progress line after every finished download (`as_completed`).
* One file should fail (for example, it is missing on the "server"). Handle the error, log it, and carry on with the others.

**Step 3 - Check and analyse with processes (calculation job)**
* `models.py`: a class `Delivery` with a **class method** `from_csv_row(row)` that checks the row - use a **regular expression** with `fullmatch` for the over (`\d+\.[1-6]`), allowed runs `0, 1, 2, 3, 4, 6` and a set of valid dismissals - and raises a custom `InvalidDeliveryError` for bad rows. A class `Player` holds the name and a list of `Delivery` objects, with methods `total_runs()`, `strike_rate()`, `average()` and `__str__`.
* `analysis.py`: `analyse_file(path)` reads the file with a **generator**, builds a `Player`, counts the bad rows and returns a normal dict: runs, balls, strike rate, average, number of 4s and 6s, fifties and hundreds (runs per match - use a dict comprehension), highest score and the number of good / bad rows. It must be at the top level of the file so it can be sent to other processes.
* Run `analyse_file` on all downloaded files with `ProcessPoolExecutor`.

**Step 4 - Report**
* Write messages with the `logging` module to `tournament.log` (successful downloads, failed downloads, bad rows per file).
* Print a leaderboard sorted by total runs, using a generator that gives one formatted line at a time, e.g.
  `1. Kohli      2914 runs  SR 138.4  avg 52.0  HS 122  (100s: 4, 50s: 11)   4987 good / 13 bad`
* Save `report.json` with the date and time (`datetime`), the results, the failed downloads and the time each step took.

**Step 5 - Compare**
* Run steps 2 and 3 also **one after another** (no threads / processes) and print a small table: normal time vs parallel time for downloading and for analysing. Explain: why do threads help in step 2 and processes in step 3?

**Bonus (uses Part B)**
* Allow at most **2 downloads at the same time** with a `Semaphore`, even if the executor has more workers.
* Use a **logger thread** that takes messages from a `queue.Queue` and writes them to the log; the rest of the program only `put()`s messages into the queue. Stop it with a sentinel.
* Let the user stop the program cleanly with `Ctrl + C` (`KeyboardInterrupt`): cancel the waiting futures and still write the log.

In [ ]:
# Mini Project